In [1]:
print("Hello World")

Hello World


In [2]:
import os
import gc

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, TensorDataset

from PIL import Image
from tqdm import tqdm

from transformers import AutoProcessor, CLIPModel

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

c:\Users\smitp\.conda\envs\Medical\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    DEVICE = torch.device("cpu")

print("Using:", DEVICE)

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
CUDA: 13.0
Using: cuda


In [4]:
DATASET_ROOT = r"./data/stage1_balanced"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

print("TRAIN:", TRAIN_DIR)
print("VAL:", VAL_DIR)
print("TEST:", TEST_DIR)

TRAIN: ./data/stage1_balanced\train
VAL: ./data/stage1_balanced\val
TEST: ./data/stage1_balanced\test


In [5]:
CLASSES = [
    "Pneumonia",
    "Corona Virus Disease",
    "Normal",
    "Tuberculosis",
]

CLASS_TO_IDX = {
    name: i
    for i, name in enumerate(CLASSES)
}

IDX_TO_CLASS = {
    i: name
    for i, name in enumerate(CLASSES)
}

NUM_CLASSES = len(CLASSES)

print(CLASS_TO_IDX)

{'Pneumonia': 0, 'Corona Virus Disease': 1, 'Normal': 2, 'Tuberculosis': 3}


In [6]:
for split_name, split_dir in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR)
]:
    
    print("\n" + "=" * 50)
    print(split_name)
    print("=" * 50)

    total = 0

    for class_name in CLASSES:

        class_dir = os.path.join(
            split_dir,
            class_name
        )

        if not os.path.exists(class_dir):
            print(f"{class_name}: FOLDER NOT FOUND")
            continue

        files = [
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            )
        ]

        print(f"{class_name}: {len(files)}")

        total += len(files)

    print("TOTAL:", total)


TRAIN
Pneumonia: 1204
Corona Virus Disease: 1218
Normal: 1207
Tuberculosis: 1220
TOTAL: 4849

VAL
Pneumonia: 400
Corona Virus Disease: 406
Normal: 402
Tuberculosis: 406
TOTAL: 1614

TEST
Pneumonia: 402
Corona Virus Disease: 407
Normal: 404
Tuberculosis: 408
TOTAL: 1621


In [7]:
MODEL_NAME = "openai/clip-vit-base-patch32"

print("Loading processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

print("Loading CLIP...")

clip_model = CLIPModel.from_pretrained(
    MODEL_NAME
)

clip_model = clip_model.to(DEVICE)
clip_model.eval()

# Freeze CLIP
for parameter in clip_model.parameters():
    parameter.requires_grad = False

print("CLIP loaded.")

Loading processor...
Loading CLIP...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 35453.61it/s]


CLIP loaded.


In [8]:
class LungImageDataset(Dataset):

    def __init__(self, root_dir):

        self.samples = []

        for class_name in CLASSES:

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            if not os.path.exists(class_dir):
                continue

            for filename in os.listdir(class_dir):

                if filename.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp")
                ):

                    image_path = os.path.join(
                        class_dir,
                        filename
                    )

                    label = CLASS_TO_IDX[class_name]

                    self.samples.append(
                        (image_path, label)
                    )

        print(
            f"{os.path.basename(root_dir)}: "
            f"{len(self.samples)} images"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(
            image_path
        ).convert("RGB")

        return image, label

In [9]:
def clip_collate_fn(batch):

    images, labels = zip(*batch)

    inputs = processor(
        images=list(images),
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"]

    labels = torch.tensor(
        labels,
        dtype=torch.long
    )

    return pixel_values, labels

In [10]:
train_dataset = LungImageDataset(
    TRAIN_DIR
)

val_dataset = LungImageDataset(
    VAL_DIR
)

test_dataset = LungImageDataset(
    TEST_DIR
)

train: 4849 images
val: 1614 images
test: 1621 images


In [11]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=clip_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=clip_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=clip_collate_fn
)

In [12]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([16, 3, 224, 224])
Labels: torch.Size([16])


In [13]:
images = images.to(DEVICE)

with torch.inference_mode():

    outputs = clip_model.get_image_features(
        pixel_values=images
    )

    features = outputs.pooler_output

print("Feature type:", type(features))
print("Feature shape:", features.shape)

Feature type: <class 'torch.Tensor'>
Feature shape: torch.Size([16, 512])


In [14]:
def extract_embeddings(loader, split_name):

    all_features = []
    all_labels = []

    print(
        f"\nExtracting {split_name} embeddings..."
    )

    clip_model.eval()

    for images, labels in tqdm(
        loader,
        desc=split_name
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        with torch.inference_mode():

            outputs = clip_model.get_image_features(
                pixel_values=images
            )

            features = outputs.pooler_output

            # Normalize CLIP embeddings
            features = features / features.norm(
                dim=-1,
                keepdim=True
            )

        all_features.append(
            features.cpu()
        )

        all_labels.append(
            labels
        )

        # Explicitly release GPU batch
        del images
        del outputs
        del features

    all_features = torch.cat(
        all_features,
        dim=0
    )

    all_labels = torch.cat(
        all_labels,
        dim=0
    )

    print(
        f"{split_name} features:",
        all_features.shape
    )

    print(
        f"{split_name} labels:",
        all_labels.shape
    )

    return all_features, all_labels

In [15]:
train_features, train_labels = extract_embeddings(
    train_loader,
    "TRAIN"
)


Extracting TRAIN embeddings...


TRAIN: 100%|██████████| 304/304 [00:56<00:00,  5.34it/s]

TRAIN features: torch.Size([4849, 512])
TRAIN labels: torch.Size([4849])


In [16]:
val_features, val_labels = extract_embeddings(
    val_loader,
    "VALIDATION"
)


Extracting VALIDATION embeddings...


VALIDATION: 100%|██████████| 101/101 [00:19<00:00,  5.20it/s]

VALIDATION features: torch.Size([1614, 512])
VALIDATION labels: torch.Size([1614])


In [17]:
test_features, test_labels = extract_embeddings(
    test_loader,
    "TEST"
)


Extracting TEST embeddings...


TEST: 100%|██████████| 102/102 [00:18<00:00,  5.38it/s]

TEST features: torch.Size([1621, 512])
TEST labels: torch.Size([1621])


In [18]:
torch.save(
    {
        "features": train_features,
        "labels": train_labels,
        "classes": CLASSES
    },
    "clip_train_embeddingsv2.pt"
)

torch.save(
    {
        "features": val_features,
        "labels": val_labels,
        "classes": CLASSES
    },
    "clip_val_embeddingsv2.pt"
)

torch.save(
    {
        "features": test_features,
        "labels": test_labels,
        "classes": CLASSES
    },
    "clip_test_embeddingsv2.pt"
)

print("Embeddings saved.")

Embeddings saved.


In [19]:
del clip_model
del processor

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("CLIP removed from GPU.")

CLIP removed from GPU.


In [20]:
train_features = train_features.to(DEVICE)
train_labels = train_labels.to(DEVICE)

val_features = val_features.to(DEVICE)
val_labels = val_labels.to(DEVICE)

test_features = test_features.to(DEVICE)
test_labels = test_labels.to(DEVICE)

print(
    "Train:",
    train_features.shape,
    train_labels.shape
)

print(
    "Val:",
    val_features.shape,
    val_labels.shape
)

print(
    "Test:",
    test_features.shape,
    test_labels.shape
)

Train: torch.Size([4849, 512]) torch.Size([4849])
Val: torch.Size([1614, 512]) torch.Size([1614])
Test: torch.Size([1621, 512]) torch.Size([1621])


In [21]:
classifier = nn.Sequential(

    nn.Linear(
        512,
        256
    ),

    nn.ReLU(),

    nn.Dropout(0.3),

    nn.Linear(
        256,
        NUM_CLASSES
    )
)

classifier = classifier.to(DEVICE)

print(classifier)

Sequential(
  (0): Linear(in_features=512, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=256, out_features=4, bias=True)
)


In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
EPOCHS = 2000

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_accuracy = 0.0

best_model_path = (
    "clip_5class_best_classifierv2.pth"
)

for epoch in range(EPOCHS):
    
    classifier.train()

    train_logits = classifier(
        train_features
    )

    train_loss = criterion(
        train_logits,
        train_labels
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    train_predictions = train_logits.argmax(
        dim=1
    )

    train_accuracy = (
        train_predictions == train_labels
    ).float().mean().item()

    classifier.eval()

    with torch.inference_mode():

        val_logits = classifier(
            val_features
        )

        val_loss = criterion(
            val_logits,
            val_labels
        )

        val_predictions = val_logits.argmax(
            dim=1
        )

        val_accuracy = (
            val_predictions == val_labels
        ).float().mean().item()

    history["train_loss"].append(
        train_loss.item()
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss.item()
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss.item():.4f} | "
        f"Train Acc: {train_accuracy*100:.2f}% | "
        f"Val Loss: {val_loss.item():.4f} | "
        f"Val Acc: {val_accuracy*100:.2f}%"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            classifier.state_dict(),
            best_model_path
        )

        print(
            f"  ✓ Saved best model: "
            f"{best_val_accuracy*100:.2f}%"
        )

print("\nTraining finished.")

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy*100:.2f}%"
)

Epoch 01/2000 | Train Loss: 1.3871 | Train Acc: 25.16% | Val Loss: 1.3810 | Val Acc: 25.15%
  ✓ Saved best model: 25.15%
Epoch 02/2000 | Train Loss: 1.3810 | Train Acc: 25.16% | Val Loss: 1.3755 | Val Acc: 25.15%
Epoch 03/2000 | Train Loss: 1.3751 | Train Acc: 26.56% | Val Loss: 1.3697 | Val Acc: 37.55%
  ✓ Saved best model: 37.55%
Epoch 04/2000 | Train Loss: 1.3695 | Train Acc: 40.81% | Val Loss: 1.3636 | Val Acc: 53.22%
  ✓ Saved best model: 53.22%
Epoch 05/2000 | Train Loss: 1.3632 | Train Acc: 53.89% | Val Loss: 1.3572 | Val Acc: 60.72%
  ✓ Saved best model: 60.72%
Epoch 06/2000 | Train Loss: 1.3565 | Train Acc: 60.94% | Val Loss: 1.3503 | Val Acc: 65.86%
  ✓ Saved best model: 65.86%
Epoch 07/2000 | Train Loss: 1.3495 | Train Acc: 64.80% | Val Loss: 1.3432 | Val Acc: 68.15%
  ✓ Saved best model: 68.15%
Epoch 08/2000 | Train Loss: 1.3421 | Train Acc: 67.73% | Val Loss: 1.3356 | Val Acc: 69.64%
  ✓ Saved best model: 69.64%
Epoch 09/2000 | Train Loss: 1.3349 | Train Acc: 68.78% | Val 

In [24]:
test_data = torch.load(
    "clip_test_embeddingsv2.pt",
    map_location="cpu"
)

In [25]:
test_features = test_data["features"].to(DEVICE)
test_labels = test_data["labels"].to(DEVICE)

In [26]:
CLASSES = [
    "Bacterial Pneumonia",
    "Corona Virus Disease",
    "Normal",
    "Tuberculosis",
    "Viral Pneumonia"
]

In [27]:
print(test_features.shape)
print(test_labels.shape)

torch.Size([1621, 512])
torch.Size([1621])


In [28]:
with torch.inference_mode():

    test_logits = classifier(
        test_features
    )

    test_predictions = test_logits.argmax(
        dim=1
    )

test_accuracy = (
    test_predictions == test_labels
).float().mean().item()

print(
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)

Test Accuracy: 96.36%


In [29]:
from sklearn.metrics import classification_report

y_true = test_labels.cpu().numpy()
y_pred = test_predictions.cpu().numpy()

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(test_data["classes"]))),
        target_names=test_data["classes"],
    )
)

                      precision    recall  f1-score   support

           Pneumonia       0.97      0.93      0.95       402
Corona Virus Disease       0.98      0.97      0.98       407
              Normal       0.93      0.97      0.95       404
        Tuberculosis       0.98      0.99      0.98       408

            accuracy                           0.96      1621
           macro avg       0.96      0.96      0.96      1621
        weighted avg       0.96      0.96      0.96      1621



In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)        

[[372   3  27   0]
 [  2 395   2   8]
 [ 10   1 392   1]
 [  1   3   1 403]]
